# Student Retention Analysis
## Statistical Analysis, Data Preparation and Modelling

**Institution:** Alberta Post-Secondary Institution (Hypothetical)  
**Dataset:** RetentionData_Final.csv  
**Cohorts:** Fall 2016 – Fall 2024 (9 years)  
**Scope:** 7 Schools · 61 Programs · 25,414 Students

---

This notebook documents the full analytical pipeline for understanding and predicting student retention at a post-secondary institution. The workflow follows the CRISP-DM framework and is structured in four main stages:

1. **Data Loading and Cleaning** — import the dataset, verify structure and completeness
2. **Exploratory Data Analysis (EDA)** — understand the distribution of students across schools, programs, pathways, and years
3. **Statistical Testing** — use chi-square tests to confirm whether observed retention gaps are statistically real
4. **Logistic Regression Modelling** — identify which factors independently drive attrition after controlling for confounding variables

Each section begins with a plain-English explanation of what the code does and why, before the code is run.


---
## Section 1: Import Libraries

We begin by importing `pandas`, the Python library used for all data manipulation throughout this notebook.

`pandas` provides the `DataFrame` structure — a two-dimensional table that mirrors what you would see in Excel, but with full programmatic control. All data loading, grouping, filtering, and transformation will be done using `pandas`.


In [1]:
import pandas as pd
import numpy as np

---
## Section 2: Load the Dataset

The dataset is loaded from a CSV file into a pandas DataFrame. Each row in the raw data represents a **group of students** sharing the same combination of school, program, national status, admission pathway, and admit term — not an individual student.

The key columns are:
- **AdmitTerm** — the fall term in which students enrolled (e.g. Fall 2016)
- **School** — one of seven anonymised schools (School A through School G)
- **Program** — one of 61 academic program codes
- **NationalStatus** — Domestic or International
- **AdmissionPathway** — how the student entered: Direct Entry, Foundation, or Transfer
- **Enrolled** — total number of students who enrolled in that group
- **Retained** — how many of those students returned the following fall term

The output shows the first and last few rows of the dataset so we can verify it loaded correctly.


In [2]:
df = pd.read_csv('RetentionData_Final.csv')
df.columns = df.columns.str.strip()
for c in df.columns:
    if df[c].dtype == object:
        df[c] = df[c].str.strip()
df

,AdmitTerm,School,Program,NationalStatus,AdmissionPathway,Enrolled,Retained
0,Fall 2016,School A,CTD,DOMESTIC,Direct Entry,199,172
1,Fall 2016,School A,CTD,DOMESTIC,Transfer,64,55
2,Fall 2016,School A,CTD,INTERNATIONAL,Direct Entry,10,9
3,Fall 2016,School A,CTD,INTERNATIONAL,Transfer,5,4
4,Fall 2016,School A,DIB,DOMESTIC,Direct Entry,26,21
...,...,...,...,...,...,...,...
1517,Fall 2024,School G,ICC,DOMESTIC,Transfer,4,2
1518,Fall 2024,School G,ICC,INTERNATIONAL,Direct Entry,3,2
1519,Fall 2024,School G,IWU,DOMESTIC,Direct Entry,10,9
1520,Fall 2024,School G,IWU,DOMESTIC,Transfer,5,4


---
## Section 3: Inspect Data Types and Completeness

`df.info()` is a standard first step in any data analysis. It tells us:

- How many rows and columns exist in the dataset
- What data type each column holds (`str` for text, `int64` for whole numbers)
- Whether any column contains missing values (shown as `Non-Null Count`)

**What we are checking for:**  
All 1,522 rows should have non-null values in every column. Any column with missing values would need to be handled before modelling — for example, by imputing a value or removing incomplete rows. Missing data in the outcome variable (`Retained`) would make analysis impossible.


In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1522 entries, 0 to 1521
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   AdmitTerm         1522 non-null   str  
 1   School            1522 non-null   str  
 2   Program           1522 non-null   str  
 3   NationalStatus    1522 non-null   str  
 4   AdmissionPathway  1522 non-null   str  
 5   Enrolled          1522 non-null   int64
 6   Retained          1522 non-null   int64
dtypes: int64(2), str(5)
memory usage: 83.4 KB


---
## Section 4: Check for Missing Values

`df.isna().sum()` counts the number of missing (null) values in each column.


In this dataset, all 1,522 rows are fully populated, so we can proceed with analysis without any imputation or row removal.


In [4]:
df.isna().sum()

AdmitTerm           0
School              0
Program             0
NationalStatus      0
AdmissionPathway    0
Enrolled            0
Retained            0
dtype: int64

---
## Section 5: List Column Names

`df.columns` prints all column names in the DataFrame. 


In [5]:
df.columns

Index(['AdmitTerm', 'School', 'Program', 'NationalStatus', 'AdmissionPathway',
       'Enrolled', 'Retained'],
      dtype='str')

---
## Section 6: Unique Admit Terms (Cohorts)

`df.AdmitTerm.unique()` lists all distinct values in the `AdmitTerm` column. This tells us how many cohort years are present in the data.

The dataset spans **nine fall cohorts** from Fall 2016 through Fall 2024. 

In [6]:
df.AdmitTerm.unique()

<StringArray>
['Fall 2016', 'Fall 2017', 'Fall 2018', 'Fall 2019', 'Fall 2020', 'Fall 2021',
 'Fall 2022', 'Fall 2023', 'Fall 2024']
Length: 9, dtype: str

---
## Section 7: Explore Unique Values Across All Columns

This loop prints the unique values and count of distinct entries for each column in the dataset. It serves several purposes:

- **Categorical columns** (School, Program, NationalStatus, AdmissionPathway): confirms how many distinct groups exist and what they are called. This is essential before any groupby analysis or encoding.
- **Numeric columns** (Enrolled, Retained): shows the range of values — useful for understanding whether any group has unusually small or large enrollment, which can affect the reliability of statistical results.

Key findings from this step:
- 9 unique admit terms (one per year)
- 7 schools
- 61 programs
- 2 national statuses: Domestic and International
- 3 admission pathways: Direct Entry, Foundation, Transfer
- Enrolled ranges from 1 to 398 students per group record


In [7]:
cols = df.columns[:-1]
for i in df[cols]:
    print(f"Column: {i}")
    print(f"Unique values: {df[i].unique()}")
    print(f"Count of unique values: {df[i].nunique()}")
    print("─" * 60)

Column: AdmitTerm
Unique values: <StringArray>
['Fall 2016', 'Fall 2017', 'Fall 2018', 'Fall 2019', 'Fall 2020', 'Fall 2021',
 'Fall 2022', 'Fall 2023', 'Fall 2024']
Length: 9, dtype: str
Count of unique values: 9
────────────────────────────────────────────────────────────
Column: School
Unique values: <StringArray>
['School A', 'School B', 'School C', 'School D', 'School E', 'School F',
 'School G']
Length: 7, dtype: str
Count of unique values: 7
────────────────────────────────────────────────────────────
Column: Program
Unique values: <StringArray>
['CTD', 'DIB', 'HBP', 'YDD', 'ZVS', 'AEX', 'CCH', 'DLF', 'ENR', 'FJC', 'GIZ',
 'JQP', 'NSU', 'UQU', 'XEM', 'AFX', 'CQK', 'FOR', 'HIQ', 'LBR', 'NOS', 'UVZ',
 'YXQ', 'CMX', 'HPA', 'IAU', 'JRA', 'JVH', 'KUY', 'LBW', 'MFQ', 'RFI', 'RNV',
 'THM', 'UKS', 'XLL', 'ZCU', 'EDB', 'FCT', 'LOC', 'NGF', 'PEW', 'UQB', 'EGJ',
 'KAU', 'KOK', 'LNT', 'PFB', 'RZP', 'SYW', 'ZSW', 'IWU', 'XJQ', 'REJ', 'QYB',
 'TWJ', 'SKM', 'ICC', 'PJN', 'RUP', 'YDA']
Length: 

In [8]:
print(f"Enrolled  — min: {df.Enrolled.min()}, max: {df.Enrolled.max()}")
print(f"Retained  — min: {df.Retained.min()}, max: {df.Retained.max()}")

Enrolled  — min: 1, max: 398
Retained  — min: 0, max: 318


---
## Section 9: Initial Exploration — Retention by Admission Pathway

This is the first analytical step. We group the data by `AdmissionPathway` and calculate total enrolled, total retained, and the retention rate for each pathway.

**Retention Rate formula:**

$$\text{Retention Rate} = \frac{\text{Retained}}{\text{Enrolled}} \times 100$$



In [9]:
df2 = df.groupby(['AdmissionPathway'])[['Enrolled','Retained']].sum()
df2['Retention Rate'] = round(df2['Retained'] / df2['Enrolled'] * 100, 1)
df2

,Enrolled,Retained,Retention Rate
AdmissionPathway,,,
Direct Entry,18083,14771,81.7
Foundation,3130,2539,81.1
Transfer,4201,3010,71.6


---
## Section 10: Retention Rates Across All Categorical Variables

This loop calculates retention rates for every categorical grouping variable in the dataset — admit term, school, program, national status, and admission pathway — in a single automated step.


**Key findings from this step:**
- **By year:** Retention dropped sharply in 2020 (77.5%) and 2021 (74.3%), likely reflecting COVID-19 disruption. It partially recovered but has not returned to the 2016 of 82.5%.
- **By school:** School G stands out at 69.8%, well below all other schools which range from 79.1% to 81.3%.
- **By program:** CCH records only 27.1% retention across nine years — below the 80.0% institutional average.
- **By national status:** International students retain at 85.5%, higher than domestic students at 79.0%.
- **By pathway:** Transfer students retain at 71.6%, the lowest of all three pathways.


In [10]:
cols = df.columns[:-1]
for i in df[cols]:
    df3 = df.groupby([i])[['Enrolled','Retained']].sum()
    df3['Retention Rate'] = round(df3['Retained'] / df3['Enrolled'] * 100, 1)
    print(df3)
    print()

           Enrolled  Retained  Retention Rate
AdmitTerm                                    
Fall 2016      3205      2643            82.5
Fall 2017      2834      2271            80.1
Fall 2018      2744      2282            83.2
Fall 2019      2836      2280            80.4
Fall 2020      2532      1962            77.5
Fall 2021      2701      2008            74.3
Fall 2022      2519      2028            80.5
Fall 2023      3067      2476            80.7
Fall 2024      2976      2370            79.6

          Enrolled  Retained  Retention Rate
School                                      
School A      8183      6470            79.1
School B      3676      2924            79.5
School C      2951      2400            81.3
School D      4651      3751            80.6
School E      1711      1370            80.1
School F      3980      3222            81.0
School G       262       183            69.8

         Enrolled  Retained  Retention Rate
Program                                    

---
## Section 11: Feature Engineering — Add RetentionRate, NotRetained, and Year

Before modelling, we add three new columns to the DataFrame:

**RetentionRate** — the percentage of enrolled students retained for each group record, calculated at the row level. This is useful for descriptive analysis and visualisation.

**NotRetained** — the number of students who did not return (Enrolled minus Retained). This is needed directly for the chi-square test, which requires both retained and not-retained counts to build a 2×2 contingency table.

**Year** — extracted from the `AdmitTerm` string (e.g. "Fall 2016" → 2016). Having year as a separate numeric or categorical column makes it easier to use in models and groupby operations.




In [11]:
df['RetentionRate'] = round((df['Retained'] / df['Enrolled']) * 100, 1)
df['NotRetained']   = df['Enrolled'] - df['Retained']
df['Year']          = df['AdmitTerm'].str.split(' ', expand=True)[1]
df

,AdmitTerm,School,Program,NationalStatus,AdmissionPathway,Enrolled,Retained,RetentionRate,NotRetained,Year
0,Fall 2016,School A,CTD,DOMESTIC,Direct Entry,199,172,86.4,27,2016
1,Fall 2016,School A,CTD,DOMESTIC,Transfer,64,55,85.9,9,2016
2,Fall 2016,School A,CTD,INTERNATIONAL,Direct Entry,10,9,90.0,1,2016
3,Fall 2016,School A,CTD,INTERNATIONAL,Transfer,5,4,80.0,1,2016
4,Fall 2016,School A,DIB,DOMESTIC,Direct Entry,26,21,80.8,5,2016
...,...,...,...,...,...,...,...,...,...,...
1517,Fall 2024,School G,ICC,DOMESTIC,Transfer,4,2,50.0,2,2024
1518,Fall 2024,School G,ICC,INTERNATIONAL,Direct Entry,3,2,66.7,1,2024
1519,Fall 2024,School G,IWU,DOMESTIC,Direct Entry,10,9,90.0,1,2024
1520,Fall 2024,School G,IWU,DOMESTIC,Transfer,5,4,80.0,1,2024


---
## Section 12: Expand Aggregated Data to Individual Level

This is one of the most important transformations in the entire pipeline, and it is worth understanding carefully.

**The problem:** Our raw data has one row per group (e.g. School A, CTD, Domestic, Direct Entry, Fall 2016 → 199 enrolled, 172 retained). Logistic regression requires one row per individual student with a binary outcome: 1 if retained, 0 if not retained.

**The solution — row expansion:**  
For each group record, we repeat the row as many times as there are enrolled students, then assign the first `Retained` rows a Target of 1 and the remaining rows a Target of 0.

**Example:**  
A group with Enrolled = 5, Retained = 3 becomes:

| School | Program | Target |
|--------|---------|--------|
| School A | CTD | 1 |
| School A | CTD | 1 |
| School A | CTD | 1 |
| School A | CTD | 0 |
| School A | CTD | 0 |

This expansion converts our 1,522 group records into **25,414 individual-level rows** — one per student — with no information added or lost. The aggregated columns (Enrolled, Retained, RetentionRate, NotRetained, AdmitTerm) are dropped since they are no longer meaningful at the individual level.

**Note on the index:** After expansion, multiple rows share the same original index number (because they came from the same group record). This is expected and does not cause problems.


In [12]:
# Repeat each row by the number of enrolled students
df_expanded = df.loc[df.index.repeat(df['Enrolled'])].copy()

# Assign Target = 1 for retained students, 0 for not retained
df_expanded['Target'] = (
    df_expanded.groupby(level=0).cumcount() < df_expanded['Retained']
).astype(int)

# Drop columns that are no longer meaningful at the individual level
df_expanded = df_expanded.drop(
    columns=['Enrolled', 'Retained', 'NotRetained', 'RetentionRate', 'AdmitTerm']
)

print(f"Expanded rows: {len(df_expanded):,}")
print(f"Retained (Target=1): {df_expanded['Target'].sum():,}")
print(f"Not Retained (Target=0): {(df_expanded['Target']==0).sum():,}")
print()
print(df_expanded.head())

Expanded rows: 25,414
Retained (Target=1): 20,320
Not Retained (Target=0): 5,094

     School Program NationalStatus AdmissionPathway  Year  Target
0  School A     CTD       DOMESTIC     Direct Entry  2016       1
0  School A     CTD       DOMESTIC     Direct Entry  2016       1
0  School A     CTD       DOMESTIC     Direct Entry  2016       1
0  School A     CTD       DOMESTIC     Direct Entry  2016       1
0  School A     CTD       DOMESTIC     Direct Entry  2016       1


---
## Section 13: Inspect the Expanded Dataset

We display the full expanded DataFrame to confirm it looks as expected. Each row now represents one student, with their school, program, national status, admission pathway, and year — and a binary Target column indicating whether they were retained (1) or not (0).



In [13]:
df_expanded

,School,Program,NationalStatus,AdmissionPathway,Year,Target
0,School A,CTD,DOMESTIC,Direct Entry,2016,1
0,School A,CTD,DOMESTIC,Direct Entry,2016,1
0,School A,CTD,DOMESTIC,Direct Entry,2016,1
0,School A,CTD,DOMESTIC,Direct Entry,2016,1
0,School A,CTD,DOMESTIC,Direct Entry,2016,1
...,...,...,...,...,...,...
1520,School G,IWU,DOMESTIC,Transfer,2024,1
1520,School G,IWU,DOMESTIC,Transfer,2024,0
1521,School G,IWU,INTERNATIONAL,Direct Entry,2024,1
1521,School G,IWU,INTERNATIONAL,Direct Entry,2024,1


---
## Section 14: Chi-Square Tests of Independence

### Why chi-square?

Looking at the retention rates in Section 10, we can see that transfer students retain at 71.6% compared to 81.7% for direct entry students. But is that a real difference, or could it just be random variation in the data?

Chi-square is a statistical test that answers exactly this question. It compares **what we actually observed** to **what we would expect to see if there were no difference between groups**. If the observed counts are far enough from the expected counts, the test concludes the difference is real — not noise.

### What the test produces

- **Rate (Grp 1) and Rate (Grp 2):** The raw retention rates being compared
- **Odds Ratio (OR):** How many times the odds of retention in Group 1 are relative to Group 2. An OR below 1.0 means Group 1 has lower odds. An OR of 0.567 means Group 1 has 56.7% of the odds — or equivalently, 44% lower odds — of being retained.
- **95% Confidence Interval:** The range we are 95% confident contains the true odds ratio. If the entire interval is below 1.0, the direction of the finding is certain regardless of exactly where the true value sits.
- **p-value:** The probability the gap happened by chance. Below 0.05 is the conventional threshold for concluding a finding is statistically significant. Below 0.001 means less than a 0.1% chance of a false positive.
- **chi2:** The test statistic — larger values indicate stronger evidence of a real difference.

### The four comparisons run

1. **Transfer vs. Direct Entry** — the primary hypothesis of the study
2. **Transfer vs. Foundation** — confirms whether the transfer gap holds against all other pathways
3. **International Transfer vs. Domestic Transfer** — tests whether national status compounds the transfer disadvantage
4. **School G vs. School A** — tests whether School G's lower rate is statistically significant, using School A (the largest school) as the reference


In [14]:
from scipy.stats import chi2_contingency

df['NotRetained'] = df['Enrolled'] - df['Retained']

def chi_square_test(df, filter_col=None, filter_val=None,
                    group_col=None, group1=None, group2=None):
    if filter_col:
        df = df[df[filter_col] == filter_val]
    g1 = df[df[group_col] == group1]
    g2 = df[df[group_col] == group2]
    r1, n1 = g1['Retained'].sum(), g1['NotRetained'].sum()
    r2, n2 = g2['Retained'].sum(), g2['NotRetained'].sum()
    rate1 = r1 / (r1 + n1)
    rate2 = r2 / (r2 + n2)
    table = [[r1, n1], [r2, n2]]
    chi2, p, dof, expected = chi2_contingency(table)
    odds1, odds2 = r1 / n1, r2 / n2
    OR = odds1 / odds2
    log_or = np.log(OR)
    se = np.sqrt(1/r1 + 1/n1 + 1/r2 + 1/n2)
    ci_low  = np.exp(log_or - 1.96 * se)
    ci_high = np.exp(log_or + 1.96 * se)
    return {
        'Rate (Grp 1)': f'{rate1*100:.1f}%',
        'Rate (Grp 2)': f'{rate2*100:.1f}%',
        'Odds Ratio':   round(OR, 3),
        '95% CI':       f'[{ci_low:.3f}, {ci_high:.3f}]',
        'p-value':      '< 0.001' if p < 0.001 else round(p, 3),
        'chi2':         round(chi2, 2)
    }

results = {}
results['Transfer vs. Direct Entry'] = chi_square_test(
    df, group_col='AdmissionPathway', group1='Transfer', group2='Direct Entry')
results['Transfer vs. Foundation'] = chi_square_test(
    df, group_col='AdmissionPathway', group1='Transfer', group2='Foundation')
results['Intl. Transfer vs. Dom. Transfer'] = chi_square_test(
    df, filter_col='AdmissionPathway', filter_val='Transfer',
    group_col='NationalStatus', group1='INTERNATIONAL', group2='DOMESTIC')
results['School G vs. School A'] = chi_square_test(
    df, group_col='School', group1='School G', group2='School A')

results_df = pd.DataFrame(results).T
results_df.index.name = 'Comparison'
print(results_df.to_string())

                                 Rate (Grp 1) Rate (Grp 2) Odds Ratio          95% CI  p-value    chi2
Comparison                                                                                            
Transfer vs. Direct Entry               71.6%        81.7%      0.567  [0.525, 0.612]  < 0.001  212.28
Transfer vs. Foundation                 71.6%        81.1%      0.588  [0.526, 0.658]  < 0.001   86.89
Intl. Transfer vs. Dom. Transfer        66.2%        72.3%      0.751  [0.610, 0.925]    0.008    7.01
School G vs. School A                   69.8%        79.1%      0.613  [0.469, 0.803]  < 0.001   12.36


---
## Section 15: Logistic Regression Model

### Why logistic regression after chi-square?

Chi-square confirmed that the transfer gap is statistically real. But it cannot answer the follow-up question: is the gap caused by the transfer pathway itself, or are transfer students simply concentrated in worse-performing schools or programs that would produce poor retention regardless of pathway?

Logistic regression solves this by controlling for all variables simultaneously. 
### How the model works

The model takes the expanded 25,414-row dataset where each student is represented individually with a Target of 1 (retained) or 0 (not retained). It learns a coefficient for every category of every input variable. These coefficients are expressed as **log-odds** — exponentiated to produce **odds ratios** which are easier to interpret.

**Reading coefficients:**
- Negative coefficient → that category reduces the odds of retention relative to the reference category
- Positive coefficient → that category increases the odds of retention
- Near zero → no meaningful difference from the reference

**Reference categories (the invisible baseline):**
- Admission Pathway: Direct Entry
- National Status: Domestic
- School: School A
- Program: AEX
- Year: 2016

All coefficients are measured relative to these baselines.



In [16]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Convert Year to string so the encoder treats it as categorical
df_expanded['Year'] = df_expanded['Year'].astype(str)

categorical_cols = ['School', 'Program', 'NationalStatus', 'AdmissionPathway', 'Year']
X = df_expanded[categorical_cols]
y = df_expanded['Target']

# ── Class balance check ───────────────────────────────────────────────────────
print("── Class Distribution ──")
class_ratio = y.value_counts(normalize=True) * 100
print(class_ratio.round(2))
minority_pct = class_ratio.min()
if minority_pct < 30:
    print(f"\n⚠️  IMBALANCED — minority class is {minority_pct:.1f}%")
else:
    print(f"\n✅  BALANCED — minority class is {minority_pct:.1f}%")

# ── Train / test split ────────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ── Pipeline ──────────────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(transformers=[
    ('ohe', OneHotEncoder(drop='first', sparse_output=False,
                          handle_unknown='ignore'), categorical_cols)
])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier',   LogisticRegression(C=1.0, max_iter=1000, random_state=42))
])

pipeline.fit(X_train, y_train)

# ── Evaluation ────────────────────────────────────────────────────────────────
y_pred = pipeline.predict(X_test)

print("\n── Model Evaluation ──")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0,
                             target_names=['Not Retained', 'Retained']))
print("Confusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm,
    index=['Actual: Not Retained', 'Actual: Retained'],
    columns=['Predicted: Not Retained', 'Predicted: Retained'])
print(cm_df)

── Class Distribution ──
Target
1    79.96
0    20.04
Name: proportion, dtype: float64

⚠️  IMBALANCED — minority class is 20.0%

── Model Evaluation ──
Accuracy: 0.8007

Classification Report:
              precision    recall  f1-score   support

Not Retained       0.62      0.02      0.03      1019
    Retained       0.80      1.00      0.89      4064

    accuracy                           0.80      5083
   macro avg       0.71      0.51      0.46      5083
weighted avg       0.76      0.80      0.72      5083

Confusion Matrix:
                      Predicted: Not Retained  Predicted: Retained
Actual: Not Retained                       16                 1003
Actual: Retained                           10                 4054


---
## Section 16: Logistic Regression Coefficients — Full Breakdown for Management

### How to read this table

Each row represents one category of one variable and shows:

- **Value** — the specific category (e.g. Transfer, School G, 2021)
- **Coefficient** — the log-odds shift associated with that category relative to its reference
- **Odds Ratio** — `e^(Coefficient)`. Values below 1.0 indicate lower retention odds; values above 1.0 indicate higher retention odds.
- **Explanation** — a plain-English interpretation of the magnitude and direction

### Risk classification used

| Odds Ratio | Classification |
|---|---|
| Below 0.50 | HIGH RISK — 50%+ lower odds |
| 0.50 – 0.70 | Moderate risk — 30–49% lower odds |
| 0.70 – 0.85 | Slight risk — 15–29% lower odds |
| 0.85 – 0.95 | Minimal risk — 5–14% lower odds |
| 0.95 – 1.05 | No meaningful difference |
| Above 1.05 | Positive — better than reference |



In [17]:
ohe_features = (pipeline.named_steps['preprocessor']
                         .named_transformers_['ohe']
                         .get_feature_names_out(categorical_cols)
                         .tolist())

coef_values = pipeline.named_steps['classifier'].coef_[0]

coef_df = pd.DataFrame({'Raw Feature': ohe_features, 'Coefficient': coef_values})

def parse_feature(raw):
    if raw.startswith('School_'):           return 'School', raw.replace('School_', '')
    elif raw.startswith('Program_'):        return 'Program', raw.replace('Program_', '')
    elif raw.startswith('NationalStatus_'): return 'National Status', raw.replace('NationalStatus_', '').title()
    elif raw.startswith('AdmissionPathway_'): return 'Admission Pathway', raw.replace('AdmissionPathway_', '')
    elif raw.startswith('Year_'):           return 'Year', raw.replace('Year_', '')
    return 'Other', raw

coef_df[['Type', 'Value']] = coef_df['Raw Feature'].apply(lambda x: pd.Series(parse_feature(x)))
coef_df['Odds Ratio'] = np.exp(coef_df['Coefficient']).round(3)

def explain(row):
    coef, OR, typ = row['Coefficient'], row['Odds Ratio'], row['Type']
    ref_map = {
        'School': 'vs. School A (reference)',
        'Program': 'vs. Program AEX (reference)',
        'National Status': 'vs. Domestic (reference)',
        'Admission Pathway': 'vs. Direct Entry (reference)',
        'Year': 'vs. 2016 (reference)',
    }
    ref = ref_map.get(typ, '')
    if abs(coef) < 0.05:
        return ref
    elif coef > 0:
        pct = round((OR - 1) * 100)
        level = 'Strong positive' if pct >= 50 else 'Moderate positive' if pct >= 20 else 'Slight positive'
        return f'{level} — {pct}% higher odds of retention {ref}'
    else:
        pct = round((1 - OR) * 100)
        level = 'HIGH RISK' if pct >= 50 else 'Moderate risk' if pct >= 30 else 'Slight risk' if pct >= 15 else 'Minimal risk'
        return f'{level} — {pct}% lower odds of retention {ref}'

coef_df['Explanation'] = coef_df.apply(explain, axis=1)

final_table = coef_df[['Type', 'Value', 'Coefficient', 'Odds Ratio', 'Explanation']].copy()
final_table['Coefficient'] = final_table['Coefficient'].round(3)
final_table = final_table.sort_values('Coefficient')

print("=" * 90)
print("LOGISTIC REGRESSION COEFFICIENTS — FULL BREAKDOWN FOR MANAGEMENT")
print("=" * 90)
print("Reference categories: School A | Program AEX | Domestic | Direct Entry | Year 2016")
print("Odds Ratio < 1.0 = higher attrition risk | Odds Ratio > 1.0 = better retention")
print("=" * 90)

for category in ['Admission Pathway', 'National Status', 'School', 'Year', 'Program']:
    subset = final_table[final_table['Type'] == category].copy()
    print(f"\n── {category.upper()} ──")
    print(subset[['Value', 'Coefficient', 'Odds Ratio', 'Explanation']].to_string(index=False))

print("\n" + "=" * 90)
print("HIGH RISK SUMMARY — Variables with strongest negative effect on retention")
print("=" * 90)
high_risk = final_table[final_table['Coefficient'] < -0.3].sort_values('Coefficient')
print(high_risk[['Type', 'Value', 'Coefficient', 'Odds Ratio', 'Explanation']].to_string(index=False))

print("\n" + "=" * 90)
print("STRONG POSITIVES — Variables with strongest positive effect on retention")
print("=" * 90)
strong_pos = final_table[final_table['Coefficient'] > 0.3].sort_values('Coefficient', ascending=False)
print(strong_pos[['Type', 'Value', 'Coefficient', 'Odds Ratio', 'Explanation']].to_string(index=False))

LOGISTIC REGRESSION COEFFICIENTS — FULL BREAKDOWN FOR MANAGEMENT
Reference categories: School A | Program AEX | Domestic | Direct Entry | Year 2016
Odds Ratio < 1.0 = higher attrition risk | Odds Ratio > 1.0 = better retention

── ADMISSION PATHWAY ──
     Value  Coefficient  Odds Ratio                                                                 Explanation
  Transfer       -0.663       0.515    Moderate risk — 48% lower odds of retention vs. Direct Entry (reference)
Foundation        0.094       1.098 Slight positive — 10% higher odds of retention vs. Direct Entry (reference)

── NATIONAL STATUS ──
        Value  Coefficient  Odds Ratio                                                             Explanation
International        0.518       1.679 Strong positive — 68% higher odds of retention vs. Domestic (reference)

── SCHOOL ──
   Value  Coefficient  Odds Ratio                                                               Explanation
School G       -0.283       0.754        Slig

---
## Summary of Findings

This notebook has completed the full analytical pipeline from raw data to statistical modelling. The key findings are:

**1. The transfer attrition gap is real and large**  
Transfer students retain at 71.6% — 10.1 percentage points below the 80.0% institutional average. The chi-square test (χ² = 212.28, p < 0.001) confirms this is not random variation. The logistic regression coefficient of −0.663 (OR = 0.515) confirms the gap persists after controlling for school, program, national status, and year simultaneously.

**2. International transfer students are the highest-risk subgroup**  
Within transfer students, international students retain at only 66.2% compared to 72.3% for domestic transfer students (OR = 0.751, p = 0.008).

**3. School G has an independent retention problem**  
School G retains at 69.8% overall. Even after the model controls for which programs and student types School G serves, the school-level coefficient remains negative (−0.283), confirming a school-specific effect that is not explained by student composition alone.

**4. Retention has declined since 2016**  
The year coefficients show a consistent downward trend since the 2016 baseline. The 2020 and 2021 cohorts (COVID-era) show the largest drops, but even 2022–2024 remain below the pre-2020 baseline.

**5. Several programs require urgent review**  
Programs CCH (27.1% raw retention), QYB, KUY, JRA, TWJ, and JVH all show high-risk coefficients in the logistic model. CCH in particular has only 48 students across nine years, making its coefficient unreliable — but the raw rate of 27.1% is independently alarming.

**6. School D and programs HPA, ENR, THM are high performers**  
These represent institutional best practices worth examining and potentially replicating.

---
*Analysis conducted on RetentionData_Final.csv · Python 3 · pandas · scikit-learn · scipy*
